# DQN Agent

Implements **Double Dueling DQN** with experience replay.

| Component | Detail |
|---|---|
| Replay Buffer | Uniform, capacity 20k |
| Target network | Hard update every 200 steps |
| Exploration | ε-greedy with exponential decay |
| Loss | Huber (SmoothL1) |
| Double DQN | Online net selects action, target net evaluates |

> Run `preprocessing.ipynb` and `pricing_env.ipynb` first to generate `data/processed/`.

In [1]:
import random
from collections import deque
from dataclasses import dataclass
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch : 2.9.1+cu128
Device  : cpu


## Q-Network

In [2]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden_dim=128, dueling=True):
        super().__init__()
        self.dueling = dueling
        self.encoder = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        if dueling:
            self.value_head     = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 1))
            self.advantage_head = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, n_actions))
        else:
            self.q_head = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, n_actions))

    def forward(self, x):
        h = self.encoder(x)
        if self.dueling:
            V = self.value_head(h)
            A = self.advantage_head(h)
            return V + (A - A.mean(dim=1, keepdim=True))
        return self.q_head(h)

## Replay Buffer

In [3]:
@dataclass
class Transition:
    state:      np.ndarray
    action:     int
    reward:     float
    next_state: np.ndarray
    done:       bool

class ReplayBuffer:
    def __init__(self, capacity=20_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size) -> List[Transition]:
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

## DQN Agent

In [4]:
class DQNAgent:
    def __init__(
        self,
        state_dim, n_actions,
        hidden_dim=128, lr=1e-3, gamma=0.99,
        epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.997,
        batch_size=64, buffer_capacity=20_000,
        target_update_freq=200, dueling=True, device="auto",
    ):
        self.device = torch.device(
            "cuda" if (device == "auto" and torch.cuda.is_available()) else "cpu"
        )
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq

        self.online_net = QNetwork(state_dim, n_actions, hidden_dim, dueling).to(self.device)
        self.target_net = QNetwork(state_dim, n_actions, hidden_dim, dueling).to(self.device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        self.loss_fn   = nn.SmoothL1Loss()
        self.buffer    = ReplayBuffer(buffer_capacity)

        self.epsilon       = epsilon_start
        self.epsilon_end   = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.steps_done    = 0

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            return int(self.online_net(s).argmax(dim=1).item())

    def push(self, state, action, reward, next_state, done):
        self.buffer.push(state, action, reward, next_state, done)

    def learn(self):
        if len(self.buffer) < self.batch_size:
            return None

        batch       = self.buffer.sample(self.batch_size)
        states      = torch.tensor(np.array([t.state      for t in batch]), dtype=torch.float32, device=self.device)
        actions     = torch.tensor([t.action              for t in batch], dtype=torch.long,    device=self.device)
        rewards     = torch.tensor([t.reward              for t in batch], dtype=torch.float32, device=self.device)
        next_states = torch.tensor(np.array([t.next_state for t in batch]), dtype=torch.float32, device=self.device)
        dones       = torch.tensor([t.done                for t in batch], dtype=torch.float32, device=self.device)

        q_values = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            best_actions = self.online_net(next_states).argmax(dim=1)
            next_q   = self.target_net(next_states).gather(1, best_actions.unsqueeze(1)).squeeze(1)
            targets  = rewards + self.gamma * next_q * (1.0 - dones)

        loss = self.loss_fn(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online_net.parameters(), max_norm=10.0)
        self.optimizer.step()

        self.steps_done += 1
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

        if self.steps_done % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())

        return float(loss.item())

    def save(self, path):
        torch.save({
            "online_net": self.online_net.state_dict(),
            "target_net": self.target_net.state_dict(),
            "optimizer":  self.optimizer.state_dict(),
            "epsilon":    self.epsilon,
            "steps_done": self.steps_done,
        }, path)
        print(f"Saved → {path}")

    def load(self, path):
        ckpt = torch.load(path, map_location=self.device)
        self.online_net.load_state_dict(ckpt["online_net"])
        self.target_net.load_state_dict(ckpt["target_net"])
        self.optimizer.load_state_dict(ckpt["optimizer"])
        self.epsilon    = ckpt["epsilon"]
        self.steps_done = ckpt["steps_done"]
        print(f"Loaded ← {path}")

print(f"DQNAgent defined. Params: {sum(p.numel() for p in QNetwork(11,5).parameters()):,}")

DQNAgent defined. Params: 34,950


## Training loop

In [5]:
def train(env, agent, n_episodes=300, log_every=10):
    history = {"episode_reward": [], "episode_revenue": [], "loss": [], "epsilon": []}

    for ep in range(1, n_episodes + 1):
        state, _ = env.reset()
        total_reward, total_revenue, losses = 0.0, 0.0, []
        done = False

        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            agent.push(state, action, reward, next_state, float(done))
            loss = agent.learn()
            if loss is not None:
                losses.append(loss)

            total_reward += reward
            if info["accepted"]:
                total_revenue += info["adjusted_price"]
            state = next_state

        history["episode_reward"].append(total_reward)
        history["episode_revenue"].append(total_revenue)
        history["loss"].append(np.mean(losses) if losses else 0.0)
        history["epsilon"].append(agent.epsilon)

        if ep % log_every == 0:
            print(
                f"Ep {ep:4d}/{n_episodes} | "
                f"Revenue: ${total_revenue:,.1f} | "
                f"Reward: {total_reward:,.1f} | "
                f"Loss: {history['loss'][-1]:.4f} | "
                f"ε: {agent.epsilon:.3f}"
            )
    return history